This notebook uses the shared random training (70%), validation (15%), and test (15%) splits prepared by `preprocessing_refined.ipynb` with seed 42. Run preprocessing first. Apply task filters and feature transformations to all three sets. Compare models on validation data, then evaluate the fitted models on test data without refitting.


In [ ]:
from pyspark.ml.classification import (
    DecisionTreeClassifier,
    LogisticRegression,
    RandomForestClassifier,
)
from pyspark.ml.feature import OneHotEncoder, StringIndexer
from pyspark.sql import functions as F
from xgboost.spark import SparkXGBClassifier

from classification_helper import (
    add_strike_zone,
    assemble,
    check_label_coverage,
    classification_report,
    evaluate_models,
    evaluate_predictions,
    feature_importance,
    perform,
    plot_feature_importance,
    print_results,
    setup_spark,
)


In [ ]:
spark = setup_spark("Classification_strikeball_noplate")

train = spark.read.parquet("./data/random_full_traintest/train")
validation = spark.read.parquet("./data/random_full_traintest/validation")
test = spark.read.parquet("./data/random_full_traintest/test")

print("Training observations:", train.count())
print("Validation observations:", validation.count())
print("Test observations:", test.count())
train.groupBy("game_year").count().orderBy("game_year").show()
validation.groupBy("game_year").count().orderBy("game_year").show()
test.groupBy("game_year").count().orderBy("game_year").show()

# Use observed plate coordinates for labels and zone comparisons.
train = train.filter(F.col("zone_observed"))
validation = validation.filter(F.col("zone_observed"))
test = test.filter(F.col("zone_observed"))


In [ ]:
train = add_strike_zone(train)
validation = add_strike_zone(validation)
test = add_strike_zone(test)


In [ ]:
check_label_coverage(train, validation, test, "pitch_type")

pitch_indexer = StringIndexer(
    inputCol="pitch_type",
    outputCol="pitch_type_idx",
)
pitch_encoder = OneHotEncoder(
    inputCol="pitch_type_idx",
    outputCol="pitch_type_ohe",
)
pitch_indexer_model = pitch_indexer.fit(train)
train = pitch_indexer_model.transform(train)
validation = pitch_indexer_model.transform(validation)
test = pitch_indexer_model.transform(test)
pitch_encoder_model = pitch_encoder.fit(train)
train = pitch_encoder_model.transform(train)
validation = pitch_encoder_model.transform(validation)
test = pitch_encoder_model.transform(test)


In [ ]:
target = "in_strike_zone"
numerical = [
    "release_speed", "release_pos_x", "release_pos_y", "release_pos_z",
    "release_spin_rate", "release_extension", "spin_axis",
    "sz_top", "sz_bot", "balls", "strikes", "vx0", "vy0",
    "vz0", "ax", "ay", "az"
]
categorical = ["stand_ohe", "p_throws_ohe", "pitch_type_ohe"]
features = numerical + categorical

_, validation = assemble(train, validation, features)
train, test = assemble(train, test, features)


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        featuresCol="features",
        labelCol=target,
        maxIter=100,
    ),
    "Decision Tree": DecisionTreeClassifier(
        featuresCol="features",
        labelCol=target,
        maxDepth=10,
        seed=42,
    ),
    "Random Forest": RandomForestClassifier(
        featuresCol="features",
        labelCol=target,
        numTrees=100,
        maxDepth=10,
        seed=42,
    )
}


In [ ]:
metrics = ["accuracy", "f1", "weightedPrecision", "weightedRecall"]
results, fit_models = perform(train, validation, models, metrics, target)
predictions = {
    name: result["predictions"]
    for name, result in results.items()
}
print_results(results)


In [ ]:
# Evaluate fitted models on test data without fitting again.
print("\nTest results")
test_results = evaluate_models(test, fit_models, metrics, target)
print_results(test_results)


In [ ]:
for name, prediction in predictions.items():
    print(f"\n{name}")
    prediction.groupBy(target, "prediction").count().orderBy(target, "prediction").show()

train.groupBy(target).count().orderBy(target).show()
test.groupBy(target).count().orderBy(target).show()


In [ ]:
class_reports = {}
for name, prediction in predictions.items():
    per_class, macro_f1 = classification_report(prediction, target)
    class_reports[name] = {"per_class": per_class, "macro_f1": macro_f1}
    print(f"\n{name}")
    per_class.show(30, truncate=False)
    print(f"Macro F1: {macro_f1:.4f}")


In [ ]:
random_forest = fit_models["Random Forest"]
rf_importance = feature_importance(random_forest, train)
print(rf_importance.head(30).to_string(index=False))

plot_feature_importance(
    rf_importance,
    title="Random Forest - Top 20 Feature Importances",
)


In [ ]:
# Evaluation metrics.
metrics = ["accuracy", "f1", "weightedPrecision", "weightedRecall"]

# Mark training and validation rows for XGBoost.
train_xgb = train.withColumn("is_validation", F.lit(False))
val_xgb = validation.withColumn("is_validation", F.lit(True))

# Combine training and validation data for early stopping.
xgb_data = train_xgb.unionByName(val_xgb)

# Configure XGBoost.
xgb = SparkXGBClassifier(
    features_col="features",
    label_col=target,
    validation_indicator_col="is_validation",
    num_workers=2,
    force_repartition=True,
    repartition_random_shuffle=True,
    max_depth=5,
    n_estimators=600,
    learning_rate=0.05,
    eval_metric="logloss",
    early_stopping_rounds=20,
)

# Fit XGBoost using validation data for early stopping.
xgb_model = xgb.fit(xgb_data)

# Predict training and validation data separately.
results_train = xgb_model.transform(train_xgb)
results_val = xgb_model.transform(val_xgb)

# Evaluate training and validation predictions.
train_scores = evaluate_predictions(results_train, metrics, target)
validation_scores = evaluate_predictions(results_val, metrics, target)
for metric in metrics:
    print(f"{metric}:")
    print(f"  Training:   {train_scores[metric]:.4f}")
    print(f"  Validation: {validation_scores[metric]:.4f}")
    print()


In [ ]:
print("\nXGBoost test results")
xgb_test_results = evaluate_models(test, {"XGBoost": xgb_model}, metrics, target)
test_pred = xgb_test_results["XGBoost"]["predictions"]
print_results(xgb_test_results)


In [ ]:
per_class, macro_f1 = classification_report(test_pred, target)
per_class.show(30, truncate=False)
print(f"\nMacro F1: {macro_f1:.4f}")
